In [1]:
from nwtrace import *
import pandas as pd
import geopandas as gpd

In [2]:
segments = gpd.read_file('data/sewer_test.geojson')
nodes = gpd.read_file('data/test_manholes.geojson')

original_segments = segments.copy()

s_id = "Sewer Gravity Asset Identification"
n_id = "Asset Identification"
ups = "Sewer Gravity Upstream Maintenance Hole"
dwns = "Sewer Gravity Downstream Maintenance Hole"


In [3]:
repaired_segments = repair.repair_segment_connections(
    segments.copy(),
    nodes,
    s_id,
    n_id,
    ups,
    dwns,
    distance_threshold=0.1
)

In [4]:
primary_dataset=segments
reference_dataset=nodes
primary_id_field=s_id
reference_id_field=n_id
connection_field=ups
distance_threshold=0.1

# get endpoints for each segment
endpoints_gdf = repair.split_segments(segments, keep_segment_geometry=False, set_from_field='from', set_to_field='to', set_role_field='role')

# split into two tables with 'from' and 'to' roles respectively
from_nodes = endpoints_gdf[endpoints_gdf['role'] == 'from']
to_nodes = endpoints_gdf[endpoints_gdf['role'] == 'to']

primary_dataset = from_nodes

In [5]:
repair_segments_a, ref_a = repair.repair_connections(
    primary_dataset=from_nodes,
    reference_dataset=nodes,
    primary_id_field=s_id,
    reference_id_field=n_id,
    connection_field=ups,
    distance_threshold=0.1
)

repair_segments_a = repair_segments_a[[s_id, ups]]

repair_segments_b, ref_b = repair.repair_connections(
    primary_dataset=to_nodes,
    reference_dataset=nodes,
    primary_id_field=s_id,
    reference_id_field=n_id,
    connection_field=dwns,
    distance_threshold=0.1
)

repair_segments_b = repair_segments_b[[s_id, dwns]]

In [6]:
ref_b

,Sewer Gravity Downstream Maintenance Hole
Sewer Gravity Asset Identification,


In [7]:
repaired_segments = segments.set_index(s_id, drop=False)

repaired_segments.update(repair_segments_a)
repaired_segments.update(repair_segments_b)

In [8]:
original_segments[original_segments[s_id] == "SL4020439"]

,fid,_id,Sewer Gravity Asset Identification,Sewer Gravity Upstream Maintenance Hole,Sewer Gravity Downstream Maintenance Hole,Sewer Gravity Twin Number,Sewer Gravity Flow Type,Sewer Gravity Structure Type,Sewer Gravity Location Description,Sewer Gravity Owned By,Sewer Gravity Managed By,Sewer Gravity Install Date,Sewer Gravity Diameter,Sewer Gravity Material,Sewer Gravity Main Shape,Sewer Gravity Measured Length,Sewer Gravity Trunk Sewer,Sewer Gravity Trunk Name,geometry
9655,112502,112502,SL4020439,JP3993505975A,MH3996605946,1,Storm,SL,CULFORD RD,City,TW Distribution and Collection,1952-01-01T00:00:00,375,CP,Circular,43.2,No,None,"MULTILINESTRING ((622041.441 4839598.457, 6220..."


In [9]:
repaired_segments[repaired_segments[s_id] == "SL4020439"]

,fid,_id,Sewer Gravity Asset Identification,Sewer Gravity Upstream Maintenance Hole,Sewer Gravity Downstream Maintenance Hole,Sewer Gravity Twin Number,Sewer Gravity Flow Type,Sewer Gravity Structure Type,Sewer Gravity Location Description,Sewer Gravity Owned By,Sewer Gravity Managed By,Sewer Gravity Install Date,Sewer Gravity Diameter,Sewer Gravity Material,Sewer Gravity Main Shape,Sewer Gravity Measured Length,Sewer Gravity Trunk Sewer,Sewer Gravity Trunk Name,geometry
Sewer Gravity Asset Identification,,,,,,,,,,,,,,,,,,,
SL4020439,112502,112502,SL4020439,MH3993505975,MH3996605946,1,Storm,SL,CULFORD RD,City,TW Distribution and Collection,1952-01-01T00:00:00,375,CP,Circular,43.2,No,None,"MULTILINESTRING ((622041.441 4839598.457, 6220..."
